## 0. Setup (run every session)
Check the GPU type below matches what this job needs, then clone + install.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('/content/COCO-Urdu'):
    !git clone -q -b v2 https://github.com/umair-hassan2/COCO-Urdu.git /content/COCO-Urdu
%cd /content/COCO-Urdu
!git pull -q
!pip install -q -e .

In [ ]:
import os
# Token source, in order: an HF_TOKEN already in the environment, else Colab Secrets
# (sidebar key icon -> add HF_TOKEN, enable for this notebook). Needs WRITE scope: every job
# pushes its outputs to the private artifacts repo via scripts/sync.py.
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
except Exception:
    pass
assert os.environ.get('HF_TOKEN'), 'No HF_TOKEN. Add it in the Colab sidebar (key icon) as HF_TOKEN.'

# huggingface_hub reads HF_TOKEN from the environment, and the `!python` cells below
# are subprocesses that inherit it -- so no notebook_login() prompt is needed.
from huggingface_hub import whoami
_me = whoami(token=os.environ['HF_TOKEN'])
print('HF auth OK:', _me['name'], '| token role:', _me.get('auth', {}).get('accessToken', {}).get('role'))


## 2. COMET-Kiwi over every caption — **L4**, ~1–1.5 units
Scores the **repaired** corpus (`gemma27b_repaired.parquet`, 317,549 captions). These scores stratify the human-study sample in `05_build_study.py`, so run this **before** step 5 on the Mac.
The Mac pushes the repaired file; the stale `gemma27b.parquet` on the hub is the pre-repair output and must not be scored.

In [ ]:
!python scripts/sync.py pull data/translations/gemma27b_repaired.parquet

In [ ]:
!python -u scripts/11_comet_full.py --systems gemma27b --path data/translations/gemma27b_repaired.parquet

### Publish the outputs
On the Mac: `python scripts/sync.py pull data/qe_full`, then `scripts/05_build_study.py`.

In [ ]:
!python scripts/sync.py push data/qe_full
